In [1]:
import pandas as pd
import requests
from io import StringIO
from datetime import datetime

In [2]:
# Get today's date in required format: yyyy_mm_dd
today = datetime.today().strftime('%Y_%m_%d')
print(today)

2026_04_23


In [3]:
# Build filename dynamically
url = f'https://www.kritische-anleger.de/downloads/festgeld_zinsentwicklung_{today}.csv'
print(url)

https://www.kritische-anleger.de/downloads/festgeld_zinsentwicklung_2026_04_23.csv


In [4]:
# Pretend to be a browser
headers = {
    "User-Agent": "Mozilla/5.0"
}
# Download file
response = requests.get(url, headers=headers)
# Raise error if still blocked
response.raise_for_status()

In [5]:
# Convert text to DataFrame
df = pd.read_csv(StringIO(response.text), sep=',')

In [6]:
df.head()

,Datum,1 Monat,2 Monate,3 Monate,6 Monate,9 Monate,1 Jahr,1.5 Jahre,2 Jahre,3 Jahre,4 Jahre,5 Jahre,6 Jahre,7 Jahre,8 Jahre,9 Jahre,10 Jahre
0,01.02.2019,0.13,0.13,0.7,0.81,0.58,1.18,0.90,1.3,1.41,1.49,1.71,1.34,1.43,1.43,1.34,1.8
1,02.02.2019,0.13,0.13,0.7,0.82,0.58,1.18,0.90,1.3,1.42,1.49,1.71,1.34,1.43,1.43,1.34,1.8
2,03.02.2019,0.13,0.13,0.7,0.82,0.58,1.18,0.88,1.3,1.42,1.49,1.71,1.34,1.43,1.43,1.34,1.8
3,04.02.2019,0.13,0.13,0.7,0.82,0.58,1.18,0.88,1.3,1.42,1.49,1.71,1.34,1.43,1.43,1.34,1.8
4,05.02.2019,0.13,0.13,0.7,0.82,0.58,1.18,0.88,1.3,1.42,1.49,1.71,1.34,1.43,1.43,1.34,1.8


In [7]:
df.dtypes

Datum         object
1 Monat      float64
2 Monate     float64
3 Monate     float64
6 Monate     float64
9 Monate     float64
1 Jahr       float64
1.5 Jahre    float64
2 Jahre      float64
3 Jahre      float64
4 Jahre      float64
5 Jahre      float64
6 Jahre      float64
7 Jahre      float64
8 Jahre      float64
9 Jahre      float64
10 Jahre     float64
dtype: object

In [8]:
# Show all column names
print("Column names:")
print(df.columns.tolist())
column = df.columns.tolist()

# Find columns convertible to date format dd.mm.yyyy
date_columns = []

for col in df.columns:
    try:
        df[col] = pd.to_datetime(df[col], format='%d.%m.%Y', errors='raise')
        date_columns.append(col)
    except:
        pass

print("\nColumns convertible to datetime:")
print(date_columns)

# Require exactly one date column
if len(date_columns) != 1:
    raise ValueError("Expected exactly one date column.")

date_col = date_columns[0]

Column names:
['Datum', '1 Monat', '2 Monate', '3 Monate', '6 Monate', '9 Monate', '1 Jahr', '1.5 Jahre', '2 Jahre', '3 Jahre', '4 Jahre', '5 Jahre', '6 Jahre', '7 Jahre', '8 Jahre', '9 Jahre', '10 Jahre']

Columns convertible to datetime:
['Datum']


In [9]:
# Check all other columns are float-convertible
float_columns = []

for col in df.columns:

    if col == date_col:
        continue

    try:
        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
            .replace(['nan', 'NaN', '', 'None', 'null'], pd.NA)
            .str.replace(',', '.', regex=False)   # German decimal comma
        )

        df[col] = pd.to_numeric(df[col], errors="coerce").astype(float)

        print(f"{col}: converted to float")
        float_columns.append(col)

    except Exception as e:
        raise ValueError(f"Column '{col}' is not float-convertible. {e}")

1 Monat: converted to float
2 Monate: converted to float
3 Monate: converted to float
6 Monate: converted to float
9 Monate: converted to float
1 Jahr: converted to float
1.5 Jahre: converted to float
2 Jahre: converted to float
3 Jahre: converted to float
4 Jahre: converted to float
5 Jahre: converted to float
6 Jahre: converted to float
7 Jahre: converted to float
8 Jahre: converted to float
9 Jahre: converted to float
10 Jahre: converted to float


In [10]:
df.to_csv(
    "termstructure.csv",
    index=False,
    sep=',',
    decimal='.',      # German decimal comma
    # sep=";",          # German CSV separator
    # decimal=",",      # German decimal comma
    # date_format="%d.%m.%Y" # German date format
)